# Homework 02: SQL Mini-Project with Sakila
## Joins, Aggregation, CTEs, Window Functions, and Performance

**Due date:** _April 3, 2026 by 11:59 PM_

**Points:** 100 total

 **Name (FIRST and LAST)** — TianJian Xing

- - -

## Overview and Instructions

**Context:** This homework builds directly on our relational database + SQL module (Days 11–19). You will use a Dockerized MySQL database (the Sakila sample database you set up previously) and a Jupyter notebook to carry out a small, portfolio-ready SQL analysis.

You will:
- Connect to a Dockerized MySQL database from this notebook.
- Design and implement several non-trivial SQL queries using Sakila.
- Use joins, grouped aggregation with `HAVING`, CTEs, window functions, and a brief performance check.
- Practice explicit validation habits (row counts, sanity checks, cross-checks).
- Produce at least one manager-style report table that could stand alone in a portfolio.
- Reflect on your SQL reasoning and debugging process.
- Log any use of AI tools in an **AI Audit Log** section.
- Create a GitHub repository to share your work.

**If you get stuck, have questions, or run into any issues as you work through this assignment**, remember to consult the PCAs, ICAs, slides from class for the **Day 11-19 class periods** -- the slides in particular often have worked examples of queries that are similar to what you'll need to write for this assignment. You are also encouraged to **come to office hours** to get direct support from your instructors! If you can't find out office hours information or need to schedule by appoint, **send us an email or catch one of us at the end of class**.

**Submission:**
- Push this completed notebook (and any small helper files) to a new private GitHub repo called `cmse492-hw02-yourMSUNetID` (make sure you use your actual MSU NetID!). **Details for this process are include in Part 10 below.**
- Submit the GitHub URL via the provided Microsoft form.
- Upload the executed notebook (with outputs) to D2L as a backup.

This notebook is designed to be a **self-contained artifact**: someone with access to your Docker image and credentials should be able to rerun your analysis and understand what you did, why you did it, and how you validated it.

**Grading breakdown:** See section headings for point values.

<div class="alert alert-attention">

**Note about AI tools:** As discussed at the beginning of the course, you are allowed to use AI tools (e.g., ChatGPT, Copilot) in responsible, transparent, and ethical ways. For this particular assignment, if you end up using AI tools for assignment, but you will be expected to document your usage in an "AI Audit Log" section near the end of this notebook. See the details in the "AI Audit Log" section below.

</div>


- - -

## 0. Setup and Database Connection

In this section you will:
- Confirm your Dockerized MySQL + Sakila setup is working.
- Establish a connection from Python to the database using `sqlalchemy`.
- Define a small helper function to run SQL and get results as a `pandas` DataFrame (make sure you look at this function so that you understand how it works and can use it in the assignment!)

**Even though you are being provided with the code cell necessary to connect to the database, you should make sure you carefully review the code and understand how it is working.** If you end up needing to connect to a SQL database on your own in the future, you should know how to do so.

**Remember**: Before you try to connect to the database, make sure your Docker container is running and that the Sakila database is properly set up inside it. If you don't recall how to do this, review the instructions from the Day 17 content and get that set up first.

If you run into an issue with this part of the assignment, contact your instructors _as soon as possible_ so we can help you get it resolved. You will not be able to complete the rest of the assignment without a working database connection, so this is a critical first step.


In [42]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Credentials for the local Docker MySQL container (must match docker-compose.yml)
uname = 'sakila'
pwd = 'p_ssW0rd'
hname = 'localhost'
dbname = 'sakila'

engine = create_engine(f'mysql+pymysql://{uname}:{pwd}@{hname}/{dbname}')
connection = engine.connect()

def run_sql(query, params=None):
    """Run a SQL query and return a pandas DataFrame.
    `query` can be a string; `params` is an optional dict for parameterized queries.
    """
    with engine.connect() as conn:
        result = conn.execute(text(query), params or {})
        df = pd.DataFrame(result.fetchall(), columns=result.keys())
    return df

# Quick test to confirm that you can connect and query the database
test_df = run_sql("SELECT * FROM film LIMIT 5;")
test_df.head()

Exception during reset or similar
Traceback (most recent call last):
  File "C:\Users\19605\AppData\Roaming\Python\Python313\site-packages\pymysql\connections.py", line 813, in _write_bytes
    self._sock.sendall(data)
    ~~~~~~~~~~~~~~~~~~^^^^^^
ConnectionAbortedError: [WinError 10053] 你的主机中的软件中止了一个已建立的连接。

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "f:\Anaconda\Lib\site-packages\sqlalchemy\pool\base.py", line 987, in _finalize_fairy
    fairy._reset(
    ~~~~~~~~~~~~^
        pool,
        ^^^^^
    ...<2 lines>...
        asyncio_safe=can_manipulate_connection,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "f:\Anaconda\Lib\site-packages\sqlalchemy\pool\base.py", line 1433, in _reset
    pool._dialect.do_rollback(self)
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "f:\Anaconda\Lib\site-packages\sqlalchemy\engine\default.py", line 700, in do_rollback
    dbapi_connection.rollback()
    ~~~~~~~~~~~~~

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,1.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2026-03-10 04:49:07
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2006-02-15 05:03:42
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,1,None,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes",2006-02-15 05:03:42
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,1,None,6,2.99,130,22.99,G,Deleted Scenes,2006-02-15 05:03:42


- - -

## 1. Framing Your Mini-Project (5 points)

Your goal is to design a **small analytic story** using the Sakila database. Think of a manager at a video rental company who wants to understand something about customers, films, inventory, stores, or revenue.

In this section, you will define:
- A brief business/analytic question or set of questions you want to answer using SQL and the audience you are imagining.
- The main tables you expect to use.
- The kinds of outputs you want (e.g., a ranked table, grouped summary, percent-of-total view).

**Note**: you might find that your question and plans evolve as you start writing SQL and seeing results. This is totally normal! Just make sure to document your initial thinking here, and then as you evolve your question and plans, make sure to update this section to reflect that evolution. Make sure to keep your initial question and plans in there as well, so that we can see how your thinking evolved.

**Another note**: You might need to spend some time exploring the Sakila schema and data to come up with a question that is interesting and feasible. You can access the DB using Adminer (via `http://localhost:8080`) to explore the schema and/or use SQL queries to poke at the data as part of your process of coming up with a good question.

If you're really struggling to come up with your "story", you can start working through the SQL exercises in the sections that follow, which require you to come up with individual questions to answer with SQL. **You might find that as you do those exercises, you see a collection of insights that your can develop into your project story.** "Reverse-engineering" your project story from the SQL exercises is a totally valid way to come up with a project question and plan!

### 1.1 Project question and plan

**Prompt:** In 1–2 paragraphs, describe your mini-project:
- What is the main question or set of questions you want to answer using SQL?
- Who is the imagined audience (e.g., store manager, marketing analyst, operations lead)?
- Which core tables do you expect to rely on (list at least 3 Sakila tables and why)?
- What kind of final table(s) or figure(s) do you expect to produce (e.g., top categories by revenue, customer segments by activity)?


 **_Write your description here._**
>
>*I will analyze which film categories generate the most rental activity and revenue in the Sakila database. The main goal is to understand which types of films perform best and whether those patterns are connected to particular stores or groups of customers.*
>
>*Several core tables could be handy, especially category, film_category, film, rental, payment, and possibly customer and store. I'll try to connect film categories to actual rentals and payments. The final output should be a  summary table showing category performance measures.*

---

## 2. Joins and Multi-Table Queries (15 points)

In this section, you will write **at least two non-trivial join queries** that connect 2–3 tables with meaningful conditions (not just a single key lookup). For each query:
- Clearly state the question it answers.
- Sketch what a single row in the result represents.
- Write and run the SQL.
- Perform at least one validation check (row counts, spot-check rows against base tables, etc.).


### 2.1 Join Query 1

**Example idea (you may adapt this or pick a different one, do not use exactly this example):**
- "For each rental in a specified 60-day period, show customer name, film title, rental date, and store city."

Based on your question and plans from the previous section, write your own question and design for this first join query. You can use the example above as a template or inspiration, but make sure to write your own unique question and design that fits with your overall mini-project. For this first join query, you're expected to clearly define the following:

1. **Question in words**: Describe what you are trying to learn in one or two sentences.
2. **Row meaning sketch**: What does each row represent? Which columns should appear?
3. **Tables and keys**: Which tables are you joining (e.g., `rental`, `customer`, `inventory`, `film`, `store`, `address`, `city`)? On which key columns?
4. **SQL query**: Write the join with clear aliases.
5. **Validation**: Perform at least one validation check, such as:
   - Compare a row count to a simpler query.
   - Spot-check an individual record across base tables.


 **2.1.a Question and design**
> 
> _Question in words_
>
>I want to understand which film categories generate the most rental activity and revenue in the Sakila database.
>
> ---
> _Row meaning sketch_
>
>Each row represents one film category.
>
>The row shows the category name, the total number of rentals associated with films in that category, the total revenue earned from those rentals, the average payment amount, and the number of unique customers who rented films from that category.
>
> ---
> _Tables and keys_
>
>This query joins the following tables:
>
>category to get the category name
>
>film_category to connect films to categories
>
>inventory to connect films to physical inventory items
>
>rental to connect inventory items to rental transactions
>
>payment to connect rentals to payment records
>
> ---
> _Join keys_
>
>category.category_id = film_category.category_id
>
>film_category.film_id = inventory.film_id
>
>inventory.inventory_id = rental.inventory_id
>
>rental.rental_id = payment.rental_id_

In [ ]:
# 2.1.b Join Query 1 - SQL

# Now that you've defined your question and design for the first join query,
# write the SQL to implement it. Make sure to use clear aliases for tables and
# columns, and to include any necessary WHERE conditions to filter the data as
# needed for your question.

join_query_1 = """
SELECT
    c.name AS category_name,
    COUNT(r.rental_id) AS total_rentals,
    SUM(p.amount) AS total_revenue,
    AVG(p.amount) AS avg_payment,
    COUNT(DISTINCT r.customer_id) AS unique_customers
FROM category AS c
JOIN film_category AS fc
    ON c.category_id = fc.category_id
JOIN inventory AS i
    ON fc.film_id = i.film_id
JOIN rental AS r
    ON i.inventory_id = r.inventory_id
JOIN payment AS p
    ON r.rental_id = p.rental_id
WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01'
GROUP BY c.name
ORDER BY total_revenue DESC;
"""


# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_join_1 = run_sql(join_query_1)
df_join_1.head()

,category_name,total_rentals,total_revenue,avg_payment,unique_customers
0,Sports,732,3310.68,4.522787,422
1,Drama,700,3031.00,4.330000,414
2,Sci-Fi,708,3018.92,4.264011,414
3,Animation,737,2934.63,3.981859,409
4,Action,711,2856.89,4.018129,422


**2.1.c Validation for Join Query 1**

Describe at least one validation check you ran and what you concluded based on it.

You might:
- Show a supporting SQL snippet in a small code cell (e.g., `COUNT(*)` comparison).
- Spot-check one ID across multiple tables and describe what you saw.


In [ ]:
# Example based on the sample query above:
# check total rentals in the same date range without joins

# validation_query_1 = """
# SELECT COUNT(*) AS n_rentals_last_60_days
# FROM rental
# WHERE rental_date BETWEEN '2005-05-01' AND '2005-06-30';
# """
# run_sql(validation_query_1)

validation_query_1 = """
SELECT COUNT(*) AS total_joined_rows
FROM film_category AS fc
JOIN inventory AS i
    ON fc.film_id = i.film_id
JOIN rental AS r
    ON i.inventory_id = r.inventory_id
JOIN payment AS p
    ON r.rental_id = p.rental_id
WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01';
>"""

run_sql(validation_query_1)

,total_joined_rows
0,10176


In [ ]:
# Put your own validation query(ies) here and explanation in the next section.


 **_Write your validation explanation here._**
>
> This validation query returned 10,176 joined rows. I think it can be considered reasonable given the expected scale of rental activity in the Sakila dataset and indicates that the joins are not causing obvious data loss (too few rows) or duplication (too many rows).

### 2.2 Join Query 2

Create a **different** join that still aligns with your project theme.

**Example ideas (again, you should adapt these to your project or create your own):**
- Customer-level summary: each row is one customer with their total rentals, number of distinct categories they rent, and home city.
- Inventory-level view: each row is one film copy with film title, store, and how many times it has been rented.

Repeat the pattern as before, clearly defining:
1. Question in words.
2. Row meaning sketch.
3. Tables and keys.
4. SQL query.
5. Validation check(s).

**2.2.a Question and design**
> 
>Question in words
>
>I want to examine whether film category performance differs by store in the Sakila database.
>
>---
>
>Row meaning sketch
>
>Each row represents one store-category combination.
>
>The row shows the store ID, the film category name, the total number of rentals for that category in that store, the total revenue earned, and the number of unique customers who rented films from that category at that store.
>
>---
>
>Tables and keys
>
>This query joins the following tables:
>
>store to identify the store
>
>inventory to connect a film copy to a store
>
>rental to connect inventory items to rental transactions
>
>payment to connect rentals to payment records
>
>film_category to connect films to categories
>
>category to get category names
>
>---
>
>Join keys:
>
>store.store_id = inventory.store_id
>
>inventory.inventory_id = rental.inventory_id
>
>rental.rental_id = payment.rental_id
>
>inventory.film_id = film_category.film_id
>
>film_category.category_id = category.category_id

In [26]:
# 2.2.b Join Query 2 - SQL
join_query_2 = """
SELECT
    s.store_id,
    c.name AS category_name,
    COUNT(r.rental_id) AS total_rentals,
    SUM(p.amount) AS total_revenue,
    COUNT(DISTINCT r.customer_id) AS unique_customers
FROM store AS s
JOIN inventory AS i
    ON s.store_id = i.store_id
JOIN rental AS r
    ON i.inventory_id = r.inventory_id
JOIN payment AS p
    ON r.rental_id = p.rental_id
JOIN film_category AS fc
    ON i.film_id = fc.film_id
JOIN category AS c
    ON fc.category_id = c.category_id
WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01'
GROUP BY s.store_id, c.name
ORDER BY s.store_id, total_revenue DESC;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_join_2 = run_sql(join_query_2)
df_join_2.head()

,store_id,category_name,total_rentals,total_revenue,unique_customers
0,1,Drama,380,1735.20,275
1,1,Sports,348,1551.52,257
2,1,Comedy,318,1536.82,245
3,1,Action,382,1524.18,292
4,1,New,314,1516.86,236


**2.2.c Validation for Join Query 2**

As before, describe your validation steps and conclusions for this second join query. You can use similar validation techniques as before or come up with new ones that fit the specific question and data you are working with.

In [13]:
# As appropriate, define helper validation query(ies) for Join Query 2
validation_query_2 = """
SELECT COUNT(*) AS total_joined_rows
FROM store AS s
JOIN inventory AS i
    ON s.store_id = i.store_id
JOIN rental AS r
    ON i.inventory_id = r.inventory_id
JOIN payment AS p
    ON r.rental_id = p.rental_id
JOIN film_category AS fc
    ON i.film_id = fc.film_id
WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01';
"""
run_sql(validation_query_2)

,total_joined_rows
0,10176


 **_Describe your validation steps and conclusions here._**
 >The validation result for Join Query 2 is also 10,176 rows, which matches the validation count from Join Query 1. This is expected because both queries are based on the same underlying rental-payment transactions in the same date range.
>
>The additional join to the store table does not change the number of transaction rows, because each inventory item belongs to exactly one store. Joining through film_category does not create extra duplication in this dataset, since each film is associated with one category. Therefore, the same set of transaction records is being counted, but grouped differently in Query 2.

---

## 3. Grouped Aggregation with HAVING (15 points)

In this section you will:
- Write at least one grouped aggregation that uses `GROUP BY` and `HAVING`.
- Apply sanity checks (e.g., sum of group counts vs. raw counts, spot-check of a group).
- Tie the result to a question your audience might actually care about.


### 3.1 Aggregation Query with HAVING (required)

**Example ideas (adapt or design your own to fit your audience and align with your mini-project):**
- "Find film categories with at least 1000 total rentals, ordered from most to least rented."
- "Find customers who have spent at least \$X in total payments."

For your query, **document**:
1. Business question in words (1–2 sentences).
2. Row meaning and expected columns in the grouped result.
3. SQL query with `GROUP BY` and `HAVING`.
4. At least two sanity checks (e.g., compare grouped count sums to raw counts, spot-check a single group).


 **3.1.a Question and design**
> 
>Business question in words
>
>I want to identify which film categories generate the highest level of rental activity and revenue in the Sakila database. Specifically in categories that have a substantial number of rentals.
>
>---
>Row meaning and output sketch
>
>Each row represents one film category.
>
>---
>The output includes:
>
>category name
>
>total number of rentals
>
>total revenue generated
>
>Only categories that meet a minimum rental threshold (using HAVING) will be included in the result.

In [27]:
# 3.1.b Aggregation with GROUP BY + HAVING

# Now that you've defined your question and design for the aggregation query,
# write the SQL to implement it. Make sure to use clear aliases for tables and
# columns, to include the appropriate GROUP BY and HAVING clauses, and to filter
# the data as needed for your question.

agg_query = """
SELECT
    c.name AS category_name,
    COUNT(r.rental_id) AS total_rentals,
    SUM(p.amount) AS total_revenue
FROM category AS c
JOIN film_category AS fc
    ON c.category_id = fc.category_id
JOIN inventory AS i
    ON fc.film_id = i.film_id
JOIN rental AS r
    ON i.inventory_id = r.inventory_id
JOIN payment AS p
    ON r.rental_id = p.rental_id
WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01'
GROUP BY c.name
HAVING COUNT(r.rental_id) >= 700
ORDER BY total_revenue DESC;
"""

# Once you've defined your query, you can uncomment/edit the lines below to run it and see the results.
df_agg = run_sql(agg_query)
display(df_agg) or df_agg.head()

,category_name,total_rentals,total_revenue
0,Sports,732,3310.68
1,Drama,700,3031.00
2,Sci-Fi,708,3018.92
3,Animation,737,2934.63
4,Action,711,2856.89


,category_name,total_rentals,total_revenue
0,Sports,732,3310.68
1,Drama,700,3031.00
2,Sci-Fi,708,3018.92
3,Animation,737,2934.63
4,Action,711,2856.89


**3.1.c Sanity checks for aggregation**

Describe at least **two** checks you performed. Ideas:
- Compare the sum of group counts to a raw `COUNT(*)` over a comparable subset.
- Focus on one group (e.g., one category or one customer), query its underlying rows, and verify the count.
- If applicable, use a `LEFT JOIN` to see categories with zero values and explain what you observe.


In [34]:
# As appropriate, write helper sanity-check queries for aggregation here and
# provide explanation in the next section.

check_query_1 = """
SELECT COUNT(r.rental_id) AS total_rentals_all
FROM rental AS r
WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01';
"""
run_sql(check_query_1)

df_agg['total_rentals'].sum()

check_query_2 = """
SELECT
    c.name AS category_name,
    COUNT(r.rental_id) AS total_rentals
FROM category AS c
JOIN film_category AS fc
    ON c.category_id = fc.category_id
JOIN inventory AS i
    ON fc.film_id = i.film_id
JOIN rental AS r
    ON i.inventory_id = r.inventory_id
WHERE c.name = 'Sports'
  AND r.rental_date BETWEEN '2005-05-01' AND '2005-08-01'
GROUP BY c.name;
"""
run_sql(check_query_2)

,category_name,total_rentals
0,Sports,732


 **_Write your sanity-check explanation here._**
 >I compared the total number of rentals from the raw dataset with the sum of rentals in the filtered result. The raw query returned 10,176 rentals, while the aggregated result sums to 3,588 rentals. This is expected because the HAVING clause filters the data to include only categories with at least 700 rentals. Therefore, the aggregated result represents a subset of the total data rather than the full dataset.
>
>Second, I performed a spot-check on an individual category (e.g., “Sports”). I ran a separate query to count the number of rentals for that category (732 rentals).

- - - 

## 4. Multi-Step Query Using a CTE (15 points)

Now you will use a **CTE (`WITH` clause)** to break a complex analysis into readable steps. You may reuse logic from earlier sections, but the CTE should add structure (e.g., pre-filtering, intermediate grouping) rather than just rename an existing query.

**Example patterns:**
- First CTE: compute customer-level rental counts and total payments.
- Second step: filter to active customers, compute ranks or thresholds on top.

Your CTE query must:
- Use at least one `WITH` clause.
- Do something non-trivial in the CTE (filtering, joining, grouping, etc.).
- Be accompanied by validation (e.g., run parts of the CTE separately, check a subset).

### 4.1 CTE design and explanation

Describe your CTE in words:
- What does the CTE compute?
- What does the outer query do on top of it?
- How does this decomposition make the logic clearer than a single big query would be?


**_Write your CTE design explanation here._**
>In this query, the CTE computes category-level rental and revenue metrics for all film categories during the selected time period. It joins category, film_category, inventory, rental, and payment, then groups the results by category to calculate total rentals and total revenue for each category.
>
>The outer query then uses the results of the CTE to filter and rank high-performing categories.
>
>This decomposition separates the aggregation step from the final filtering and presentation step to make the logic clearer than writing everything as one large query.The CTE acts as an intermediate summary table, which makes the query easier to read, validate, and modify.

### 4.2 Writing the CTE query

Now that you have a plan for your CTE, write the SQL query that implements it.

In [35]:
# 4.2 CTE-based query

# Now that you've designed your CTE, write the SQL query that implements it.
# Make sure to use clear aliases for tables and columns, and to structure the
# CTE in a way that breaks down the logic into understandable steps.

cte_query = """
WITH category_summary AS (
    SELECT
        c.name AS category_name,
        COUNT(r.rental_id) AS total_rentals,
        SUM(p.amount) AS total_revenue
    FROM category AS c
    JOIN film_category AS fc
        ON c.category_id = fc.category_id
    JOIN inventory AS i
        ON fc.film_id = i.film_id
    JOIN rental AS r
        ON i.inventory_id = r.inventory_id
    JOIN payment AS p
        ON r.rental_id = p.rental_id
    WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01'
    GROUP BY c.name
)

SELECT
    category_name,
    total_rentals,
    total_revenue
FROM category_summary
WHERE total_rentals >= 700
ORDER BY total_revenue DESC;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_cte = run_sql(cte_query)
display(df_cte) or df_cte.head()

,category_name,total_rentals,total_revenue
0,Sports,732,3310.68
1,Drama,700,3031.00
2,Sci-Fi,708,3018.92
3,Animation,737,2934.63
4,Action,711,2856.89


,category_name,total_rentals,total_revenue
0,Sports,732,3310.68
1,Drama,700,3031.00
2,Sci-Fi,708,3018.92
3,Animation,737,2934.63
4,Action,711,2856.89


### 4.3 CTE validation

Run at least one validation step focused on the **intermediate CTE logic**, such as:
- Select a subset of the CTE output (e.g., `LIMIT 10`) and manually compare to base tables.
- Recompute a simpler version of a metric (e.g., total payments) for one customer and confirm it matches the CTE result.

In [36]:
# As appropriate, write helper queries to inspect the CTE result or base tables
check_cte_row = """
WITH category_summary AS (
    SELECT
        c.name AS category_name,
        COUNT(r.rental_id) AS total_rentals,
        SUM(p.amount) AS total_revenue
    FROM category AS c
    JOIN film_category AS fc
        ON c.category_id = fc.category_id
    JOIN inventory AS i
        ON fc.film_id = i.film_id
    JOIN rental AS r
        ON i.inventory_id = r.inventory_id
    JOIN payment AS p
        ON r.rental_id = p.rental_id
    WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01'
    GROUP BY c.name
)
SELECT *
FROM category_summary
WHERE category_name = 'Sports';
"""

run_sql(check_cte_row)

,category_name,total_rentals,total_revenue
0,Sports,732,3310.68


 **_Describe your validation and what you concluded here. Include helper queries in code cells if needed._**
 > I selected a specific category (Sports) from the CTE output and compared it with a direct aggregation query on the base tables.
>
>The CTE returned the Sports category with 732 total rentals and 3310.68 in total revenue. I then ran a separate query on the underlying tables, filtering for the Sports category within the same date range. The results from this direct computation matched exactly with the CTE output.

---

## 5. Window Function with `OVER (...)` (15 points)

Next, you will write at least one query that uses a **window function**. Options include:
- `ROW_NUMBER()`, `RANK()` or `DENSE_RANK()` within a category (e.g., top customers per store).
- A percent-of-total calculation using `SUM(...) OVER ()` or `SUM(...) OVER (PARTITION BY ...)`.

Your window query should:
- Use `OVER (...)` in a meaningful way.
- Produce a result that could help your audience understand ranking or relative importance.
- Include at least one validation step (e.g., verify that percentages sum to ~100%).


### 5.1 Window function design

Describe your plan:
- What are you ranking or computing percentages over?
- What does each row represent in the final result?
- How will your audience interpret the window column(s)?


 **_Write your window function plan here._**
 >I want to measure the relative importance of each film category by comparing its total revenue to the overall revenue generated during the selected time period.
>
>Each row in the final result represents one film category. For each category, the query will show its total revenue, its rank based on revenue, and its percentage contribution to total revenue across all categories.
>
>The window function allows me to compare each category to the full dataset without collapsing the result into a single summary value. The ranking helps identify the top-performing categories, while the percentage-of-total calculation helps show how much each category contributes to the overall business.

### 5.2 Writing the window function query

Now that you have a plan for your window function, write the SQL query that implements it.

In [37]:
# 5.2 Window function query

# Now that you've designed your window function query, write the SQL to
# implement it. Make sure to use clear aliases for tables and columns, to
# structure the query in a way that breaks down the logic into understandable
# steps, and to include the appropriate window function(s) with meaningful
# OVER(...) clauses.

window_query = """
WITH category_revenue AS (
    SELECT
        c.name AS category_name,
        SUM(p.amount) AS total_revenue
    FROM category AS c
    JOIN film_category AS fc
        ON c.category_id = fc.category_id
    JOIN inventory AS i
        ON fc.film_id = i.film_id
    JOIN rental AS r
        ON i.inventory_id = r.inventory_id
    JOIN payment AS p
        ON r.rental_id = p.rental_id
    WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01'
    GROUP BY c.name
)
SELECT
    category_name,
    total_revenue,
    RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
    ROUND(
        100.0 * total_revenue / SUM(total_revenue) OVER (),
        2
    ) AS pct_of_total_revenue
FROM category_revenue
ORDER BY revenue_rank, category_name;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_window = run_sql(window_query)
display(df_window) or df_window.head()

,category_name,total_revenue,revenue_rank,pct_of_total_revenue
0,Sports,3310.68,1,7.73
1,Drama,3031.00,2,7.08
2,Sci-Fi,3018.92,3,7.05
3,Animation,2934.63,4,6.85
4,Action,2856.89,5,6.67
5,Comedy,2764.10,6,6.45
6,Games,2732.97,7,6.38
7,Family,2722.01,8,6.36
8,New,2694.19,9,6.29
9,Documentary,2692.25,10,6.29


,category_name,total_revenue,revenue_rank,pct_of_total_revenue
0,Sports,3310.68,1,7.73
1,Drama,3031.00,2,7.08
2,Sci-Fi,3018.92,3,7.05
3,Animation,2934.63,4,6.85
4,Action,2856.89,5,6.67


### 5.3 Window function validation

Identify at least one validation you can perform, such as:
- Summing `pct_of_total` and confirming it is close to 100 (allowing for rounding).
- Checking that the ranking order matches the sorted totals.
- Comparing one row’s percent-of-total to a hand calculation.

In [40]:
# As appropriate, write helper queries or pandas checks for window function results
# e.g., df_window['pct_of_total'].sum()

check_window = """
WITH category_revenue AS (
    SELECT
        c.name AS category_name,
        SUM(p.amount) AS total_revenue
    FROM category AS c
    JOIN film_category AS fc
        ON c.category_id = fc.category_id
    JOIN inventory AS i
        ON fc.film_id = i.film_id
    JOIN rental AS r
        ON i.inventory_id = r.inventory_id
    JOIN payment AS p
        ON r.rental_id = p.rental_id
    WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01'
    GROUP BY c.name
),
category_pct AS (
    SELECT
        category_name,
        total_revenue,
        100.0 * total_revenue / SUM(total_revenue) OVER () AS pct_of_total_revenue
    FROM category_revenue
)
SELECT
    ROUND(SUM(pct_of_total_revenue), 2) AS total_percentage
FROM category_pct;
"""

run_sql(check_window)

,total_percentage
0,100.00


 **_Write your validation explanation here and include helper code if needed._**
>To validate the window function results, I checked whether the percentage of total revenue across all categories sums to approximately 100%.
>
>And the result was exactly 100.00, confirming that the SUM(...) OVER () window function correctly computes the total revenue across all categories and that the percentage-of-total calculation is accurate.

---

## 6. Performance Check with `EXPLAIN` and/or Index (5 points)

Pick **one** of your more complex queries from the earlier sections (join, aggregation, or window-based) and perform a brief performance analysis using MySQL's `EXPLAIN` statement.

You should:
- State which query you are examining and what aspect of its execution plan you want to inspect.
- Run `EXPLAIN` on the query and capture the output in a DataFrame.
- Identify at least one aspect of the plan (e.g., table scan vs. index usage, join order, estimated rows) and explain why it seems reasonable or concerning.
- Optionally, propose or implement a simple index and compare the `EXPLAIN` output before vs. after.

You do **not** need to time queries precisely or chase microseconds. The focus is on reading and interpreting the query plan.

**Note**: You might not see any "red flags" in the `EXPLAIN` output, especially on a small dataset like Sakila. That's totally fine! The point is to practice reading the plan and thinking about what it means.

### 6.1 Choose a query and plan your analysis

Before running `EXPLAIN`, take a moment to describe your plan:
1. Which query from an earlier section are you examining (e.g., "Section 3 aggregation by category", "Section 5 window query")?
2. Why did you choose this query (e.g., it joins many tables, it aggregates a large dataset)?
3. What do you expect to see in the `EXPLAIN` output (e.g., index usage on join keys, a full scan on a particular table)?

 **_Write your plan here._**
>I will examine the Section 5 window function query that computes total revenue by film category, ranks categories by revenue, and calculates each category’s percentage of total revenue.
>
>It is one of the more complex queries in the project, which joins several tables, performs aggregation, uses a CTE, and applies window functions in the outer query.
>
>I expect to see index usage on the join keys such as category_id, film_id, inventory_id, and rental_id. I also expect that MySQL may perform some extra work for the aggregation and sorting steps.

### 6.2 Run EXPLAIN on your chosen query

Now that you have a plan, run `EXPLAIN` on the query you chose. Note that you can prefix any SQL query with `EXPLAIN` to get the execution plan — `run_sql()` works with `EXPLAIN` statements just like any other query.

In [43]:
# 6.2 Run EXPLAIN on your chosen query

# Prefix your query with EXPLAIN to capture the execution plan.
# Note: run_sql() works with EXPLAIN statements just like any other query.

explain_query = """
WITH category_revenue AS (
    SELECT
        c.name AS category_name,
        SUM(p.amount) AS total_revenue
    FROM category AS c
    JOIN film_category AS fc
        ON c.category_id = fc.category_id
    JOIN inventory AS i
        ON fc.film_id = i.film_id
    JOIN rental AS r
        ON i.inventory_id = r.inventory_id
    JOIN payment AS p
        ON r.rental_id = p.rental_id
    WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01'
    GROUP BY c.name
)
SELECT
    category_name,
    total_revenue,
    RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
    ROUND(
        100.0 * total_revenue / SUM(total_revenue) OVER (),
        2
    ) AS pct_of_total_revenue
FROM category_revenue
ORDER BY revenue_rank, category_name;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_explain = run_sql(explain_query)
display(df_explain) or df_explain.head()

,category_name,total_revenue,revenue_rank,pct_of_total_revenue
0,Sports,3310.68,1,7.73
1,Drama,3031.00,2,7.08
2,Sci-Fi,3018.92,3,7.05
3,Animation,2934.63,4,6.85
4,Action,2856.89,5,6.67
5,Comedy,2764.10,6,6.45
6,Games,2732.97,7,6.38
7,Family,2722.01,8,6.36
8,New,2694.19,9,6.29
9,Documentary,2692.25,10,6.29


,category_name,total_revenue,revenue_rank,pct_of_total_revenue
0,Sports,3310.68,1,7.73
1,Drama,3031.00,2,7.08
2,Sci-Fi,3018.92,3,7.05
3,Animation,2934.63,4,6.85
4,Action,2856.89,5,6.67


### 6.3 Interpret the EXPLAIN output

Now that you have the `EXPLAIN` output, interpret it in a few sentences. Address at least one of the following:
- Is MySQL using an index or doing a full table scan on any of the tables? Does this seem reasonable given the query structure?
- What is the estimated number of rows being scanned for each table (the `rows` column)? Does this seem appropriate?
- Is the join order what you would expect? Why or why not?
- Based on the plan, would you suggest any changes to improve performance (e.g., adding an index, restructuring the query)?

In [ ]:
# Optional: use this cell for any helper queries or pandas checks that
# support your interpretation of the EXPLAIN output.
# For example, you could check the actual row count of a table where EXPLAIN
# shows a large row estimate.

 **_Write your performance interpretation here (2–5 sentences)._**
>I examined the output for the window function query that computes category-level revenue, ranking, and percentage contributions.
>
>From the execution plan, MySQL performs a series of joins across the category, film_category, inventory, rental, and payment tables. The join operations appear to use appropriate keys on foreign key relationships such as category_id, film_id, inventory_id, and rental_id.
>
>The estimated number of rows processed at each step appears reasonable given the relatively small size of the Sakila dataset. Even if some tables are scanned fully, this does not significantly impact performance in this context.
>
>Overall, the execution plan is consistent with the structure of the query.

### 6.4 Optional: Propose or add an index

If you'd like to go further and it seems like an optimization is needed, you can propose or create a simple index and compare the `EXPLAIN` output before vs. after. Document:
- Which column(s) you chose and why (e.g., frequently used in `WHERE` or join conditions).
- Whether the `EXPLAIN` output changed in a way that suggests better performance (e.g., less full-table scanning, lower estimated rows).

In [45]:
# Optional: create an index and re-run EXPLAIN to compare.
# create_index_sql = """
# -- TODO: Replace with your CREATE INDEX statement.
# -- Example: CREATE INDEX idx_rental_date ON rental(rental_date);
# """
# run_sql(create_index_sql)

# explain_after_index = """
# -- TODO: Replace with your EXPLAIN query after adding the index.
# """
# run_sql(explain_after_index)

---

## 7. Manager-Style Report Table (10 points)

Design **one table** that you would feel comfortable showing to your imagined audience or including in a portfolio. This table should:
- Be relatively **wide** (e.g., 4–8 columns) with clear, human-readable column labels.
- Summarize something decision-relevant (e.g., categories by revenue and share, top customers per store).
- Optionally use `CASE` expressions or window functions to create computed or pivot-like columns.

You may reuse or adapt one of your earlier queries if it already reads like a manager report, or you may write a dedicated query here.

### 7.1 Describe your report

Before writing the SQL, describe your report:
1. Who is the audience for this table (e.g., store manager, marketing analyst, operations lead)?
2. What decision or question does this table support?
3. Which columns will appear, and what does each one tell the audience?
4. Will you reuse a query from an earlier section or write a new one?

 **_Write your audience, decision, and column plan here._**
>The audience for this report is a business manager or operations lead who wants to understand which film categories generate the most revenue and how important each category is to the overall business.
>
>This table supports decisions related to content strategy and inventory planning. It helps identify which categories are the most profitable and which categories contribute less to total revenue.
>
>The report will include the following columns:
>
>- Category name: identifies each film category.
>
>- Total revenue: shows the total revenue generated by each category.
>
>- Revenue rank: ranks categories from highest to lowest revenue.
>
>- Percentage of total revenue: shows each category’s contribution to overall revenue.
>
>- Performance tier: classifies categories into High, Medium, or Low performance groups.
>
>I will reuse and extend the window function query from Section 5, since it already computes revenue, ranking, and percentage, and is suitable for a manager-style report.

### 7.2 Write the report query

Now write the SQL query that produces your report table. Aim for clear, human-readable column aliases and a layout that could stand on its own in a portfolio or presentation.

In [46]:
# 7.2 Report query SQL

# Now that you've described your report, write the SQL that produces it.
# Aim for clear column aliases and a result that could stand on its own in
# a portfolio or presentation.

report_query = """
WITH category_revenue AS (
    SELECT
        c.name AS category_name,
        SUM(p.amount) AS total_revenue
    FROM category AS c
    JOIN film_category AS fc
        ON c.category_id = fc.category_id
    JOIN inventory AS i
        ON fc.film_id = i.film_id
    JOIN rental AS r
        ON i.inventory_id = r.inventory_id
    JOIN payment AS p
        ON r.rental_id = p.rental_id
    WHERE r.rental_date BETWEEN '2005-05-01' AND '2005-08-01'
    GROUP BY c.name
),
ranked AS (
    SELECT
        category_name,
        total_revenue,
        RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
        ROUND(
            100.0 * total_revenue / SUM(total_revenue) OVER (),
            2
        ) AS pct_of_total_revenue
    FROM category_revenue
)
SELECT
    category_name,
    total_revenue,
    revenue_rank,
    pct_of_total_revenue,
    CASE
        WHEN revenue_rank <= 5 THEN 'High'
        WHEN revenue_rank <= 10 THEN 'Medium'
        ELSE 'Low'
    END AS performance_tier
FROM ranked
ORDER BY revenue_rank;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_report = run_sql(report_query)
display(df_report) or df_report.head()

,category_name,total_revenue,revenue_rank,pct_of_total_revenue,performance_tier
0,Sports,3310.68,1,7.73,High
1,Drama,3031.00,2,7.08,High
2,Sci-Fi,3018.92,3,7.05,High
3,Animation,2934.63,4,6.85,High
4,Action,2856.89,5,6.67,High
5,Comedy,2764.10,6,6.45,Medium
6,Games,2732.97,7,6.38,Medium
7,Family,2722.01,8,6.36,Medium
8,New,2694.19,9,6.29,Medium
9,Documentary,2692.25,10,6.29,Medium


,category_name,total_revenue,revenue_rank,pct_of_total_revenue,performance_tier
0,Sports,3310.68,1,7.73,High
1,Drama,3031.00,2,7.08,High
2,Sci-Fi,3018.92,3,7.05,High
3,Animation,2934.63,4,6.85,High
4,Action,2856.89,5,6.67,High


### 7.3 Optional: polish in pandas

If you wish, you can use `pandas` to make the report output more presentation-ready:
- Rename columns for readability.
- Format numeric columns (e.g., round percentages, add currency formatting).
- Select or reorder columns for clarity.

This is optional but can help the output feel more portfolio-ready.

In [24]:
# Optional: light pandas polishing for the report table, then display it.
# For example, you could format the revenue column as currency, or add a total row at the bottom.

# Put your Pandas commands to polish the report with pandas here (example: format revenue as currency):


# Display the polished report
# display(df_report)

---

## 8. Reflection on SQL Reasoning and Debugging (5 points)

Write **1–2 paragraphs** reflecting on your process. Address at least:
- How you broke down your mini-project into smaller SQL questions, or if your mini-project emerged/evolved as you explored the data and wrote individual queries.
- Any bugs, confusing results, or dead ends you hit and how you debugged them (e.g., row-count checks, `LIMIT`, CTE inspection, switching between Adminer and this notebook).
- Any tradeoffs you noticed between expressing logic with CTEs, `GROUP BY`, and window functions (e.g., readability vs. performance, ease of validation) either for this specific assignment or in general based on your experience.


 **_Write your reflection here._**
>I approached the analysis by breaking down the overall goal into smaller SQL tasks.
>
>Starting with basic joins to understand how the tables are connected, and then moving to aggregation queries to compute meaningful metrics such as total rentals and revenue by category. 
>
>As the analysis progressed, I refined these queries using HAVING clauses, CTEs, and window functions to produce more structured and insightful results.
>
>I encountered a few issues, such as mismatches between aggregated totals and raw counts, initially I thought were errors but later realized were expected due to filtering conditions like HAVING clauses.
>
>I used validation techniques such as row-count checks and subset queries. 
>
>I also learned that while CTEs improve readability and make validation easier, they can sometimes introduce additional execution overhead compared to simpler queries. Window functions provided powerful insights like ranking and percentage contributions, but required careful validation to ensure correctness.


---

## 9. AI Audit Log (5 points)

This section is about **transparent, professional use of AI tools** (including systems like ChatGPT, GitHub Copilot, Perplexity, or others). You will not lose points for using AI, but you **must** document how you used it.

Answer the prompts below honestly. If you did **not** use any AI tools, you can say so explicitly.


### 9.1 Tools used

- List any AI tools you used while working on this assignment (e.g., ChatGPT, Copilot, Perplexity, Claude, etc.). If none, write "None".
- For each tool, briefly describe what you asked it to help with (e.g., "helped me remember `CREATE INDEX` syntax", "suggested a pattern for a window function").

 **_Describe your AI usage here._**

>I used ChatGPT as a support tool while working on this assignment. Specifically, I used it to help recall SQL syntax (such as CREATE INDEX and window functions), to suggest query structures for joins, CTEs, and window functions, and to provide explanations when I encountered confusing results.
>
>It helped me understand how to structure queries and think through the logic of multi-step SQL analysis through sector 1-4.
>
>I also used Gemini to help me verify the feasibility of the objectives for each sector.
>
>It proved particularly effective during Sectors 6 and 7, as I needed to look back and ensure that the work I had completed in the preceding sectors aligned with the requirements of the current one.

### 9.2 Directly used vs. adapted

- Did you paste any AI-generated SQL or code directly into your notebook with minimal changes? If so, point to where (e.g., "Section 5 window query") and **make sure you add a comment in the code cell acknowledging the AI contribution**.
- For code or queries that you **adapted**, briefly describe what you changed and why (e.g., "adjusted table names to match Sakila, simplified to match our validation habits").

 **_Describe AI "copy-paste" usage here._**
 > None of those code above are copy-paste from AI tool.

### 9.3 Validation and trust

- How did you validate any AI-suggested queries or patterns before trusting them? (e.g., row-count checks, comparing against a simpler query, sanity checking outputs.)
- Are there any parts of your notebook where you are **not fully confident** in the result? If so, describe them, why you're lacking confidence, and what additional checks you would run if you had more time (e.g., "I wasn't sure if the window function was partitioned correctly, I would want to double-check the logic and maybe test it on a smaller dataset if I had more time").

 **_Describe your trust in AI outputs here._**
>I didn't perform extensive checks at the code level, considering that most of it was written manually (and required two or three rounds of revision just to get it working). However, I did cross-check the underlying concepts while discussing the direction of the assignment with the AI; I posed similar questions to both GPT and Gemini, and then selected the most logical ideas among their responses to serve as a starting point for implementation.

- - -

## 10. Creating your GitHub repo and pushing your code (10 points)

Now that you've finished your notebook, it's time to create a GitHub repository and push your code. 

**The goal for this section is to build a repository that someone could clone, spin up the Docker container to access the SQL database, and execute your notebook from top to bottom.**

To achieve this, follow these steps:

1. Create a new private repository on GitHub named `cmse492-hw02-yourMSUNetID` (replace `yourMSUNetID` with your actual MSU NetID).
2. Initialize the repository with a README file.
3. Clone the repository to your local machine.
4. Copy your completed notebook into the cloned repository folder. **Commit the fully-executed version so that someone could see the output _without_ having to run it, but using the Docker set up they should be able to execute it, if needed.**
5. Add the necessary Docker compose file needed to run the Sakila database (you can reuse the one provided in the course materials).
6. Update the README so that it provides an overview of the project, what you accomplished and the skills you're showcasing, and instructions on how to run the notebook and connect to the database (make sure you include information for starting up the Docker containers!).

    - You should be able to share this repository with a friend, co-worker, or future employer and they should be able to understand what you did, why you did it, and how to run your notebook and see your analysis in action.
    
7. Use Git commands to add, commit, and push your changes to GitHub. 

- - -
After you're made sure the GitHub repo is set up correctly, **Submit this notebook to D2L** and **Fill out the MS Form below** to finalize your submission.  
(If the form won't render in your notebook for some reason, you can also access it directly here: https://forms.office.com/r/Ppk0tnVkc8)

In [25]:
from IPython.display import HTML
HTML(
"""
<iframe 
	src="https://forms.office.com/r/Ppk0tnVkc8" 
	width="800" 
	height="800px" 
	frameborder="0" 
	marginheight="0" 
	marginwidth="0">
	Loading...
</iframe>
"""
)

>- - -
### Congratulations, you're done!

Submit this assignment by uploading it to the course Desire2Learn web page.  Go to the "Homework" section, find the appropriate submission link, and upload it there. **Make sure your instructors have access to your private GitHub repo by adding them as collaborators!**

See you in class!

>---
&#169; Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.